In [1]:
# -*- coding: utf-8 -*-
"""
Unified Benchmarking: 3D DAE Training, Vectorized Reconstruction, and LHS Error Analysis.
✅ Fully GPU-native evaluation on 4D LHS points (t, x, y, z).
✅ Strict CUDA synchronization and zero-overhead loss tracking.
✅ Integrated LHS 13000-point exact sampling and error computation.
✅ 1D/2D-aligned metrics logging and safety checks.
"""

import time
import os
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import grad
from scipy.stats import qmc
from scipy.spatial import cKDTree
import torch.nn.init as init
from numpy.polynomial.legendre import leggauss

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
PI = torch.tensor(np.pi, dtype=torch.float32, device=DEVICE)

# =============================================================================
# 1. 实验参数与预分配
# =============================================================================
SEEDS = [33, 99, 202, 1234, 5678, 9999]
EPOCHS = 16000
MU_LIST = [0.01]

NUM_SAMPLES = 13000
LHS_SEED = 1234

EVAL_WARMUP = 20
EVAL_REPEAT = 200

BASE_PATH = '.'

# 高斯积分节点初始化
GAUSS_NODES, GAUSS_WEIGHTS = leggauss(17)
GAUSS_NODES = torch.tensor(GAUSS_NODES, dtype=torch.float32, device=DEVICE).unsqueeze(0)
GAUSS_WEIGHTS = torch.tensor(GAUSS_WEIGHTS, dtype=torch.float32, device=DEVICE).unsqueeze(0)

# =============================================================================
# 2. 定义网络与物理函数
# =============================================================================
class PINN(nn.Module):
    def __init__(self, layers):
        super(PINN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i + 1]))
        self.apply(self._initialize_weights)
        self.activation = nn.Tanh()

    def _initialize_weights(self, layer):
        if isinstance(layer, nn.Linear):
            init.xavier_normal_(layer.weight)
            if layer.bias is not None:
                init.zeros_(layer.bias)

    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = self.activation(self.layers[i](x))
        return self.layers[-1](x)

def differentiable_gaussian_quadrature(func, a, b):
    s = (b - a) / 2 * GAUSS_NODES + (a + b) / 2
    f_s = func(s)
    integral = (b - a) / 2 * torch.sum(GAUSS_WEIGHTS * f_s, dim=1, keepdim=True)
    return integral

def varphi_minus_torch(x, y, z):
    lower_bound = -torch.ones_like(x)
    def integrand_for_batch(s_nodes): 
        x_b = x.expand(-1, s_nodes.shape[1])
        y_b = y.expand(-1, s_nodes.shape[1])
        z_b = z.expand(-1, s_nodes.shape[1])
        return torch.cos(PI * s_nodes) * torch.cos(PI * (s_nodes + y_b - x_b)) * torch.cos(PI * (s_nodes + z_b - x_b))
    integral = differentiable_gaussian_quadrature(integrand_for_batch, lower_bound, x)
    value_under_sqrt = 16 + 2 * integral
    return -torch.sqrt(torch.relu(value_under_sqrt) + 1e-8)

def varphi_plus_torch(x, y, z):
    upper_bound = torch.ones_like(x)
    def integrand_for_batch(s_nodes):
        x_b = x.expand(-1, s_nodes.shape[1])
        y_b = y.expand(-1, s_nodes.shape[1])
        z_b = z.expand(-1, s_nodes.shape[1])
        return torch.cos(PI * s_nodes) * torch.cos(PI * (s_nodes + y_b - x_b)) * torch.cos(PI * (s_nodes + z_b - x_b))
    integral = differentiable_gaussian_quadrature(integrand_for_batch, x, upper_bound)
    value_under_sqrt = 4 - 2 * integral
    return torch.sqrt(torch.relu(value_under_sqrt) + 1e-8)

def compute_loss(model, y_internal, z_internal, t_internal, y_boundary, z_boundary, t_boundary):
    u_internal = t_internal * model(torch.cat([y_internal, z_internal, t_internal], dim=1))
    
    u_y = grad(u_internal.sum(), y_internal, create_graph=True, retain_graph=True)[0]
    u_z = grad(u_internal.sum(), z_internal, create_graph=True, retain_graph=True)[0]
    u_t = grad(u_internal.sum(), t_internal, create_graph=True, retain_graph=True)[0]
    
    varphi_minus_vals = varphi_minus_torch(u_internal, y_internal, z_internal)
    varphi_plus_vals = varphi_plus_torch(u_internal, y_internal, z_internal)
    
    residual = u_t - 0.5 * (u_y + u_z - 1) * (varphi_minus_vals + varphi_plus_vals)
    loss_pde = (residual ** 2).mean()

    u_left = t_boundary * model(torch.cat([torch.ones_like(y_boundary) * (-1), z_boundary, t_boundary], dim=1))
    u_right = t_boundary * model(torch.cat([torch.ones_like(y_boundary) * (1), z_boundary, t_boundary], dim=1))
    u_bottom = t_boundary * model(torch.cat([y_boundary, torch.ones_like(z_boundary) * (-1), t_boundary], dim=1))
    u_top = t_boundary * model(torch.cat([y_boundary, torch.ones_like(z_boundary) * 1, t_boundary], dim=1))
    loss_boundary = ((u_bottom - u_top) ** 2 + (u_left - u_right) ** 2).mean()
    
    return loss_pde + loss_boundary

def dae_reconstruct_gpu(net, x_eval, y_eval, z_eval, t_eval, mu, eval_mode=False):
    """Fully GPU-native 3D DAE reconstruction."""
    if eval_mode:
        with torch.enable_grad():
            y_in = y_eval.detach().clone().requires_grad_(True)
            z_in = z_eval.detach().clone().requires_grad_(True)
            t_in = t_eval.detach().clone()
            y_z_t = torch.cat([y_in, z_in, t_in], dim=1)
            h0_val = t_in * net(y_z_t)
            
            # 联合求导释放计算图
            dh_grads = grad(h0_val.sum(), (y_in, z_in), create_graph=False, retain_graph=False)
            h0_y, h0_z = dh_grads[0].detach(), dh_grads[1].detach()
            h0 = h0_val.detach()
    else:
        with torch.enable_grad():
            y_in = y_eval.clone().requires_grad_(True)
            z_in = z_eval.clone().requires_grad_(True)
            t_in = t_eval.clone()
            y_z_t = torch.cat([y_in, z_in, t_in], dim=1)
            h0 = t_in * net(y_z_t)
            
            dh_grads = grad(h0.sum(), (y_in, z_in), create_graph=True, retain_graph=True)
            h0_y, h0_z = dh_grads[0], dh_grads[1]

    with torch.no_grad():
        phi_m_x = varphi_minus_torch(x_eval, y_eval, z_eval)
        phi_p_x = varphi_plus_torch(x_eval, y_eval, z_eval)
        phi_m_h = varphi_minus_torch(h0, y_eval, z_eval)
        phi_p_h = varphi_plus_torch(h0, y_eval, z_eval)
        
        delta_phi = phi_p_h - phi_m_h
        B_term = 1.0 - h0_y - h0_z
        
        exponent_left = torch.clamp((h0 - x_eval) * delta_phi * B_term / (2.0 * mu), -500.0, 500.0)
        U_left = phi_m_x + delta_phi / (torch.exp(exponent_left) + 1.0)
        
        exponent_right = torch.clamp((x_eval - h0) * (delta_phi) * B_term / (2.0 * mu), -500.0, 500.0)
        U_right = phi_p_x - delta_phi / (torch.exp(exponent_right) + 1.0)
        
        return torch.where(x_eval <= h0, U_left, U_right)

# =============================================================================
# 3. LHS Sampling & Error Methods
# =============================================================================
def get_target_col(df):
    if 'u' in df.columns: return 'u'
    if 'u0' in df.columns: return 'u0'
    return df.columns[-1]

def load_true_solution(mu):
    # 根据 3D 习惯，加载包含真实解的文件
    filename = f'3d_U0_true_mu{mu:.0e}.csv'
    path = os.path.join(BASE_PATH, filename)
    if not os.path.exists(path):
        filename = '3d_U0_all_t_u_x_y_z_t_mu2_51_mathematica_619.csv'
        path = os.path.join(BASE_PATH, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Cannot find true solution file in {BASE_PATH}.")
            
    df = pd.read_csv(path)
    df.columns = [str(col).lower().strip() for col in df.columns]
    df = df.sort_values(by=['t', 'x', 'y', 'z']).reset_index(drop=True)
    return df, filename

def build_or_load_lhs_test_set(mu):
    df_true, filename = load_true_solution(mu)
    index_file = f"3d_LHS_sample_indices_mu{mu:.0e}.npy"

    if os.path.exists(index_file):
        sample_indices = np.load(index_file)
        valid = (len(sample_indices) == NUM_SAMPLES and 
                 len(np.unique(sample_indices)) == NUM_SAMPLES and
                 np.min(sample_indices) >= 0 and 
                 np.max(sample_indices) < len(df_true))
        if valid:
            print(f"[mu={mu}] Loaded valid LHS indices from {index_file}.")
        else:
            sample_indices = None
    else:
        sample_indices = None

    if sample_indices is None:
        total_points = len(df_true)
        if total_points < NUM_SAMPLES:
            raise ValueError(f"Reference grid only has {total_points} points.")
            
        t_min, t_max = df_true['t'].min(), df_true['t'].max()
        x_min, x_max = df_true['x'].min(), df_true['x'].max()
        y_min, y_max = df_true['y'].min(), df_true['y'].max()
        z_min, z_max = df_true['z'].min(), df_true['z'].max()

        all_points = df_true[['t', 'x', 'y', 'z']].values
        kdtree = cKDTree(all_points)
        selected = set()
        
        batch_id = 0
        while len(selected) < NUM_SAMPLES and batch_id < 100:
            sampler = qmc.LatinHypercube(d=4, seed=LHS_SEED + batch_id)
            lhs_sample = sampler.random(n=NUM_SAMPLES)
            lhs_sample_scaled = qmc.scale(lhs_sample, [t_min, x_min, y_min, z_min], [t_max, x_max, y_max, z_max])
            _, candidate_indices = kdtree.query(lhs_sample_scaled)
            for idx in candidate_indices:
                selected.add(int(idx))
                if len(selected) == NUM_SAMPLES: break
            batch_id += 1
            
        sample_indices = np.array(list(selected))
        np.save(index_file, sample_indices)
        print(f"[mu={mu}] Regenerated & saved {NUM_SAMPLES} unique LHS indices.")

    t_lhs_np = df_true.iloc[sample_indices]['t'].values.reshape(-1, 1)
    x_lhs_np = df_true.iloc[sample_indices]['x'].values.reshape(-1, 1)
    y_lhs_np = df_true.iloc[sample_indices]['y'].values.reshape(-1, 1)
    z_lhs_np = df_true.iloc[sample_indices]['z'].values.reshape(-1, 1)
    
    true_col = get_target_col(df_true)
    true_lhs_np = df_true.iloc[sample_indices][true_col].values.reshape(-1)

    return {
        "t_lhs": torch.tensor(t_lhs_np, dtype=torch.float32, device=DEVICE),
        "x_lhs": torch.tensor(x_lhs_np, dtype=torch.float32, device=DEVICE),
        "y_lhs": torch.tensor(y_lhs_np, dtype=torch.float32, device=DEVICE),
        "z_lhs": torch.tensor(z_lhs_np, dtype=torch.float32, device=DEVICE),
        "true_lhs_np": true_lhs_np,
        "t_np": t_lhs_np, "x_np": x_lhs_np, "y_np": y_lhs_np, "z_np": z_lhs_np,
        "n_test": len(sample_indices)
    }

def compute_error(true_u, pred_u):
    diff = pred_u - true_u
    e2 = np.linalg.norm(diff) / np.linalg.norm(true_u)
    einf = np.max(np.abs(diff))
    return e2, einf

def mean_std(values):
    arr = np.asarray(values, dtype=float)
    if len(arr) <= 1: return np.nanmean(arr), 0.0
    return np.nanmean(arr), np.nanstd(arr, ddof=1)

# =============================================================================
# 4. 主程序 (Main Execution)
# =============================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print(f"Starting 3D DAE benchmark with strict LHS-based T_eval and error")
    print(f"Device: {DEVICE} | LHS test points: {NUM_SAMPLES} | LHS seed: {LHS_SEED}")
    print("="*80 + "\n")

    lhs_data = {}
    for mu in MU_LIST:
        lhs_data[mu] = build_or_load_lhs_test_set(mu)

    metrics_results = {mu: [] for mu in MU_LIST}

    for seed in SEEDS:
        print(f"\n--- Running Seed: {seed} ---")
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
            
        n_internal, n_periodic = 5000, 4800
        y_internal = (torch.rand(n_internal, 1) * 2 - 1).to(DEVICE).requires_grad_(True)
        z_internal = (torch.rand(n_internal, 1) * 2 - 1).to(DEVICE).requires_grad_(True)
        t_internal = (torch.rand(n_internal, 1) * 0.5).to(DEVICE).requires_grad_(True)
        
        y_boundary = (torch.rand(n_periodic, 1) * 2 - 1).to(DEVICE)
        z_boundary = (torch.rand(n_periodic, 1) * 2 - 1).to(DEVICE)
        t_boundary = (torch.rand(n_periodic, 1) * 0.5).to(DEVICE)

        total_points = n_internal + 4 * n_periodic
        model = PINN([3, 10, 10, 10, 10, 10, 10, 1]).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        loss_list = []

        # --- PHASE A: T_train ---
        model.train()
        if torch.cuda.is_available(): torch.cuda.synchronize()
        t_train_start = time.perf_counter()
        
        for epoch in range(EPOCHS):
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(model, y_internal, z_internal, t_internal, y_boundary, z_boundary, t_boundary)
            loss.backward()
            optimizer.step()
            
            loss_list.append(loss.detach())

        if torch.cuda.is_available(): torch.cuda.synchronize()
        T_train = time.perf_counter() - t_train_start
        
        loss_history = torch.stack(loss_list).cpu().numpy().astype(np.float64)
        e_loss = float(loss_history[-1])

        T_train_per_iter_ms = (T_train * 1e3) / EPOCHS
        T_train_per_iter_point_us = (T_train * 1e6) / (EPOCHS * total_points)
        print(f"  > Network Trained! T_train: {T_train:.2f}s | e_loss: {e_loss:.3e}")

        np.save(f'3d_DAE_loss_history_seed{seed}.npy', loss_history)

        # --- PHASE B: T_eval and Error Check ---
        model.eval()
        
        for mu in MU_LIST:
            d_mu = lhs_data[mu]
            x_lhs, y_lhs, z_lhs, t_lhs = d_mu["x_lhs"], d_mu["y_lhs"], d_mu["z_lhs"], d_mu["t_lhs"]
            true_u = d_mu["true_lhs_np"]
            n_test = d_mu["n_test"]

            # GPU Warmup
            with torch.no_grad():
                for _ in range(EVAL_WARMUP):
                    _ = dae_reconstruct_gpu(model, x_lhs, y_lhs, z_lhs, t_lhs, mu, eval_mode=True)

            if torch.cuda.is_available(): torch.cuda.synchronize()
            eval_start = time.perf_counter()

            # Repeated Timing Block (Strictly limited to LHS Points)
            with torch.no_grad():
                for _ in range(EVAL_REPEAT):
                    _ = dae_reconstruct_gpu(model, x_lhs, y_lhs, z_lhs, t_lhs, mu, eval_mode=True)

            if torch.cuda.is_available(): torch.cuda.synchronize()
            T_eval = (time.perf_counter() - eval_start) / EVAL_REPEAT

            # Error Computation
            with torch.no_grad():
                u_eval_tensor = dae_reconstruct_gpu(model, x_lhs, y_lhs, z_lhs, t_lhs, mu, eval_mode=True)
            u_pred_lhs = u_eval_tensor.detach().cpu().numpy().reshape(-1)
            
            e2, einf = compute_error(true_u, u_pred_lhs)
            T_total = T_train + T_eval
            
            metrics_results[mu].append({
                "Seed": seed, "N_test": n_test, "e_loss": e_loss, "e2": e2, "einf": einf,
                "T_train": T_train, "T_eval": T_eval, "T_total": T_total,
                "T_train_per_iter_ms": T_train_per_iter_ms,
                "T_train_per_iter_point_us": T_train_per_iter_point_us,
                "total_trained_steps": EPOCHS, "total_point_steps": EPOCHS * total_points,
                "final_residual_points": total_points, "eval_warmup": EVAL_WARMUP, "eval_repeat": EVAL_REPEAT
            })
            
            print(f"    -> [mu={mu}] T_eval={T_eval:.6e}s | e2={e2:.3e} | einf={einf:.3e}")
            
            # --- PHASE C: Saving output data strictly identical to 2D standard ---
            df_lhs_pred = pd.DataFrame({
                "t": d_mu["t_np"].reshape(-1), "x": d_mu["x_np"].reshape(-1), 
                "y": d_mu["y_np"].reshape(-1), "z": d_mu["z_np"].reshape(-1),
                "u": u_pred_lhs
            })
            df_lhs_pred.to_csv(f"3d_DAE_U0_predicted_LHS_mu{mu:.0e}_seed{seed}.csv", index=False)

# =============================================================================
# 5. 输出统计与全面打印
# =============================================================================
    print("\n" + "="*80)
    print("ALL SEEDS COMPLETED. GENERATING PUBLICATION TABLES...")
    print("="*80 + "\n")

    for mu in MU_LIST:
        df_mu = pd.DataFrame(metrics_results[mu])
        df_mu.to_csv(f'3d_DAE_mu{mu:.0e}_Metrics_Summary.csv', index=False)
        
        cols = ["e_loss", "e2", "einf", "T_train", "T_eval", "T_total", 
                "T_train_per_iter_ms", "T_train_per_iter_point_us",
                "total_trained_steps", "total_point_steps", "final_residual_points", "N_test"]
        stats = {col: mean_std(df_mu[col].values) for col in cols}
        
        print(f"### Results for 3D DAE, mu={mu} [Mean \pm Sample Std] ###")
        print(f"N_test: {stats['N_test'][0]:.0f} \pm {stats['N_test'][1]:.0f}")
        print(f"e_loss: {stats['e_loss'][0]:.3e} \pm {stats['e_loss'][1]:.3e}")
        print(f"e_2: {stats['e2'][0]:.3e} \pm {stats['e2'][1]:.3e}")
        print(f"e_inf: {stats['einf'][0]:.3e} \pm {stats['einf'][1]:.3e}")
        print(f"T_train (s): {stats['T_train'][0]:.2f} \pm {stats['T_train'][1]:.2f}")
        print(f"T_eval (s): {stats['T_eval'][0]:.6e} \pm {stats['T_eval'][1]:.6e}")
        print(f"T_total (s): {stats['T_total'][0]:.2f} \pm {stats['T_total'][1]:.2f}")
        print(f"T_train/iter (ms): {stats['T_train_per_iter_ms'][0]:.4f} \pm {stats['T_train_per_iter_ms'][1]:.4f}")
        print(f"T_train/(iter*pt) (us): {stats['T_train_per_iter_point_us'][0]:.4f} \pm {stats['T_train_per_iter_point_us'][1]:.4f}")
        print(f"Optimization steps: {stats['total_trained_steps'][0]:.1f} \pm {stats['total_trained_steps'][1]:.1f}")
        print(f"Point-iterations: {stats['total_point_steps'][0]:.1f} \pm {stats['total_point_steps'][1]:.1f}")
        print(f"Final residual points: {stats['final_residual_points'][0]:.1f} \pm {stats['final_residual_points'][1]:.1f}\n")


Starting 3D DAE benchmark with strict LHS-based T_eval and error
Device: cuda | LHS test points: 13000 | LHS seed: 1234

[mu=0.01] Loaded valid LHS indices from 3d_LHS_sample_indices_mu1e-02.npy.

--- Running Seed: 33 ---
  > Network Trained! T_train: 451.78s | e_loss: 1.193e-05
    -> [mu=0.01] T_eval=3.577361e-03s | e2=7.262e-03 | einf=1.462e+00

--- Running Seed: 99 ---
  > Network Trained! T_train: 449.68s | e_loss: 9.148e-06
    -> [mu=0.01] T_eval=3.684875e-03s | e2=6.720e-03 | einf=1.394e+00

--- Running Seed: 202 ---
  > Network Trained! T_train: 448.61s | e_loss: 1.003e-05
    -> [mu=0.01] T_eval=3.745228e-03s | e2=7.977e-03 | einf=1.659e+00

--- Running Seed: 1234 ---
  > Network Trained! T_train: 448.87s | e_loss: 1.688e-05
    -> [mu=0.01] T_eval=3.573586e-03s | e2=9.152e-03 | einf=1.998e+00

--- Running Seed: 5678 ---
  > Network Trained! T_train: 449.95s | e_loss: 3.919e-06
    -> [mu=0.01] T_eval=3.342760e-03s | e2=4.819e-03 | einf=9.203e-01

--- Running Seed: 9999 ---
